# matmul-2d — ex2: diagnose and fix a shape mismatch with a single transpose

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `matmul-2d`. Running the final beacon cell reports progress against the `Numpy: matmul 2-D` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: matmul 2-D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`matmul-2d`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "matmul-2d"
DD_SUBTOPIC = "Numpy: matmul 2-D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Diagnose-and-fix matmul mismatch via transpose — quick refresher

Two 2-D tensors are matmul-compatible iff their inner dims match: `(M, K) @ (K, N)`. If you receive `a: (M, K)` and `b: (N, K)`, the inner dims are `K` vs `N` — wrong unless you fix `b` first. Two rescue moves:

```
a @ b.T          # (M, K) @ (K, N) -> (M, N)
a.T @ b          # only valid if M == N — different output
```

**Diagnosis rule.** Read both shapes; find which axis pair already agrees in size; transpose the operand that needs to swap so the matching axis lands on the inner side.

**Exemplar.** `a` is `(3, 5)`, `b` is `(7, 5)`. The `5`s already agree, but they live on the WRONG sides (`a`'s last vs `b`'s last). Transpose `b` to `(5, 7)`, then `a @ b.T` gives `(3, 7)`.

### Exercise 2 — diagnose and fix a shape mismatch with a single transpose

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a pair of 2-D shapes that fails the matmul inner-dim rule, identify which operand needs transposing, and return the corrected product.
> Keywords: matmul, transpose, shape-debug, inner-dim-fix
> ```

**KCs targeted:** `matmul-2d-shape-rule`, `transpose-to-align-inner-dim`

Implement `ex2_fix_matmul(a, b)`. Given two 2-D tensors `a` and `b` (each `(rows, cols)`), apply the following rules and return the result:

1. If `a.shape[1] == b.shape[0]`, no fix needed — return `a @ b`.
2. Else if `a.shape[1] == b.shape[1]` (i.e. `b`'s LAST dim matches `a`'s last dim), transpose `b` and return `a @ b.T`.
3. Else if `a.shape[0] == b.shape[0]` (i.e. `a`'s FIRST dim matches `b`'s first dim), transpose `a` and return `a.T @ b`.
4. Otherwise no single transpose fixes the mismatch — return `None`.

**Rule order matters.** Check the no-fix case first, then the two single-transpose cases in the stated order. If multiple branches would fire (e.g. `a` and `b` are both square and symmetric on size), the earlier rule wins.

**Examples.**
- `a: (3, 5), b: (5, 7)` → already compatible. Return `(3, 7)` product.
- `a: (3, 5), b: (7, 5)` → inner mismatch but `a.shape[1] == b.shape[1] == 5`. Return `a @ b.T` of shape `(3, 7)`.
- `a: (3, 5), b: (3, 7)` → inner mismatch but `a.shape[0] == b.shape[0] == 3`. Return `a.T @ b` of shape `(5, 7)`.
- `a: (3, 5), b: (8, 11)` → no shared axis. Return `None`.

**The test uses concrete random tensors** and verifies both the chosen transpose AND the numerical equivalence to a manually-applied `t.matmul` on the correctly-shaped operands.

In [ ]:
def ex2_fix_matmul(a: Tensor, b: Tensor):
    # Rule 1: already compatible.
    if a.shape[1] == b.shape[0]:
        return a @ b
    # Rule 2: a's last dim matches b's last dim → transpose b.
    if a.shape[1] == b.shape[1]:
        return a @ b.T
    # Rule 3: a's first dim matches b's first dim → transpose a.
    if a.shape[0] == b.shape[0]:
        return a.T @ b
    # Rule 4: no single transpose fixes it.
    return None


<details><summary>Solution</summary>

```python
def ex2_fix_matmul(a: Tensor, b: Tensor):
    # Rule 1: already compatible.
    if a.shape[1] == b.shape[0]:
        return a @ b
    # Rule 2: a's last dim matches b's last dim → transpose b.
    if a.shape[1] == b.shape[1]:
        return a @ b.T
    # Rule 3: a's first dim matches b's first dim → transpose a.
    if a.shape[0] == b.shape[0]:
        return a.T @ b
    # Rule 4: no single transpose fixes it.
    return None
```

**Why each rule exists.** Matmul needs `a.shape[1] == b.shape[0]`. There are exactly two ways a single `.T` can create that match: transpose `b` to swap its dims (covers the case `a.shape[1] == b.shape[1]`); transpose `a` (covers `a.shape[0] == b.shape[0]`). If neither shared axis exists, no single transpose works — you'd need a reshape, broadcast, or different operands.

**Why rule-order matters.** When multiple rules would fire, the result depends on which you check first — and the choice is semantically meaningful. `a @ b.T` and `a.T @ b` produce DIFFERENT tensors (different shapes, different values). Documenting the priority is the only way the function behaves predictably.

**This is the real debugging move.** When you get `RuntimeError: mat1 and mat2 shapes cannot be multiplied`, this is the diagnostic loop: print both shapes, find the shared axis, transpose the wrong operand. The drill is the shape-debug-as-function-call version of that workflow.

**Why we don't always succeed.** `(3, 5)` and `(8, 11)` share nothing — neither transpose helps. The caller would have to look elsewhere (broadcasting, einsum with explicit contractions, or a genuine bug in their pipeline).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()